# Exercise 15.2: Numerically Stable Softmax & Log-Sum-Exp

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/Maths-for-IT-Notebooks/blob/main/Chapter-15/02-stable-softmax-logsumexp.ipynb)

**Reference:** Chapter 15 — Derivatives, Chain Rule and Numerical Stability (`chapters/derivatives-chain-rule-stability.tex`)  
**Estimated Time:** 15–20 min  

### Learning Objectives
- Observe floating-point overflow when computing naive $\operatorname{softmax}(z)$ on large logits.
- Implement the max-shift trick $\operatorname{softmax}(z_i) = \frac{e^{z_i - m}}{\sum_j e^{z_j - m}}$ where $m = \max_j z_j$.
- Implement the numerically stable $\operatorname{logsumexp}(z)$ function used in cross-entropy losses.


In [ ]:
import numpy as np


### Mathematical Background

In classification and LLM next-token prediction, logits $\mathbf{z} \in \mathbb{R}^k$ are converted to probabilities via **softmax**:

$$\operatorname{softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^k e^{z_j}}$$

When logits are large (e.g., `[1000, 1001, 1002]`), $e^{1000}$ overflows `float64` to `inf`, producing `inf / inf = nan`.

Subtracting $m = \max_j z_j$ leaves the mathematical probabilities unchanged while guaranteeing every exponent $z_j - m \le 0$:

$$\operatorname{softmax}(z_i) = \frac{e^{z_i - m}}{\sum_{j=1}^k e^{z_j - m}}, \qquad \operatorname{logsumexp}(\mathbf{z}) = m + \ln\left(\sum_{j=1}^k e^{z_j - m}\right)$$


In [ ]:
def naive_softmax(z: np.ndarray) -> np.ndarray:
    """Compute softmax directly without max-shifting (prone to overflow).

    Formula:
        softmax(z)_i = exp(z_i) / sum_j exp(z_j)

    Args:
        z: 1D NumPy array of logits.

    Returns:
        1D NumPy array of probabilities (or NaNs on overflow).
    """
    # TODO: Compute np.exp(z) / np.sum(np.exp(z))
    raise NotImplementedError("Complete this TODO")


def stable_softmax(z: np.ndarray) -> np.ndarray:
    """Compute numerically stable softmax using the max-shift trick.

    Formula:
        m = max(z)
        softmax(z)_i = exp(z_i - m) / sum_j exp(z_j - m)

    Args:
        z: 1D NumPy array of logits.

    Returns:
        1D NumPy array of valid probabilities summing to 1.0.
    """
    # TODO: Subtract np.max(z) before exponentiating
    raise NotImplementedError("Complete this TODO")


def logsumexp(z: np.ndarray) -> float:
    """Compute log(sum(exp(z))) in a numerically stable way.

    Formula:
        logsumexp(z) = m + log(sum_j exp(z_j - m)), where m = max(z)

    Args:
        z: 1D NumPy array of logits.

    Returns:
        Scalar float equal to log(sum(exp(z))).
    """
    # TODO: Implement stable log-sum-exp and return a Python float
    raise NotImplementedError("Complete this TODO")


<details>
<summary>Hint</summary>

- In `stable_softmax(z)`, compute `m = np.max(z)` and exponentiate `z - m`.
- Because at least one entry of `z - m` is `0.0`, `exp(0.0) = 1.0`, so the denominator is always $\ge 1.0$ (never 0) and every exponent is $\le 0$ (never overflows).
- In `logsumexp(z)`, return `float(m + np.log(np.sum(np.exp(z - m))))`.

</details>


In [ ]:
# 1. Figure 15.1 example: logits [1000, 1001] -> [0.269, 0.731]
p_fig = stable_softmax(np.array([1000.0, 1001.0]))
assert np.allclose(p_fig, [0.26894142, 0.73105858], atol=1e-6)

# 2. Large logits [1000, 1001, 1002]: naive overflows to NaN, stable succeeds
large_logits = np.array([1000.0, 1001.0, 1002.0])
with np.errstate(over='ignore', invalid='ignore'):
    p_naive = naive_softmax(large_logits)
assert np.all(np.isnan(p_naive)), 'Naive softmax should produce NaNs on [1000, 1001, 1002]'

p_stable = stable_softmax(large_logits)
p_small = stable_softmax(np.array([0.0, 1.0, 2.0]))
assert np.all(np.isfinite(p_stable))
assert np.isclose(np.sum(p_stable), 1.0)
assert np.allclose(p_stable, p_small)

# 3. Very negative logits [-1000, -1001, -1002]: no underflow-to-zero division
neg_logits = np.array([-1000.0, -1001.0, -1002.0])
p_neg = stable_softmax(neg_logits)
assert np.allclose(p_neg, stable_softmax(np.array([2.0, 1.0, 0.0])))

# 4. LogSumExp identity: softmax(z) == exp(z - logsumexp(z))
lse = logsumexp(large_logits)
assert np.isfinite(lse)
assert np.isclose(lse, 1000.0 + logsumexp(np.array([0.0, 1.0, 2.0])))
assert np.allclose(np.exp(large_logits - lse), p_stable)

print('All Chapter 15.2 assertions passed!')


### Think Deeper

1. Why does subtracting a constant $m$ from every logit $z_i$ leave $\operatorname{softmax}(\mathbf{z})$ mathematically unchanged?
2. How does the max-shift trick also protect against **underflow** when all logits are very negative (e.g., `[-1000, -1001, -1002]`)?
